In [1]:
import pandas as pd

from semevalpolar.utils import get_project_root
import os
from semevalpolar.llm.data_utils import read_dataset, create_submission, split_dataframe
from semevalpolar.finetuning.slm.inference import run_inference_on_df

from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer

D:\projects\mldl\semevalpolar


In [4]:
project_root = get_project_root()
df_data = read_dataset(os.path.join(get_project_root(), 'data', 'dev_phase', 'subtask1', 'train', 'eng.csv'))

In [3]:
polarized = df_data[df_data["polarization"] == 1]
not_polarized = df_data[df_data["polarization"] == 0]

polarized = polarized.sample(n=len(not_polarized), random_state=42)
balance_df = (pd.concat([polarized, not_polarized], ignore_index=True)).sample(n=len(polarized) * 2, random_state=42)

In [ ]:
balance_df

In [ ]:
train_df, val_df, test_df = split_dataframe(df_data)

train_df = train_df.rename(columns={"polarization": "label"})
val_df = val_df.rename(columns={"polarization": "label"})
test_df = test_df.rename(columns={"polarization": "label"})

for df in [train_df, val_df, test_df]:
    df["label"] = df["label"].astype(int)

In [ ]:
from datasets import DatasetDict, Dataset

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

dataset = dataset.remove_columns("__index_level_0__")

In [ ]:
model_dir = os.path.join(
    project_root,
    "predictions",
    "finetuning",
    "saved_model",
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

dataset_test = dataset["test"].map(tokenize_fn, batched=True)

In [ ]:
trainer = Trainer(model=model)

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

predictions = trainer.predict(dataset_test)
logits = predictions.predictions
labels = predictions.label_ids
preds = np.argmax(logits, axis=-1)

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")